In [ ]:
# Cell 1
# Mục đích: đọc cả 9 file Olist vào bộ nhớ, toàn bộ dạng chữ, để các cell sau chỉ việc kiểm tra.
import os
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv

load_dotenv()                                   # đọc file .env, nạp vào biến môi trường
OLIST_DIR = Path(os.getenv("OLIST_DIR", ""))    # đường dẫn thư mục Olist; "" nếu chưa có
print("Thư mục:", OLIST_DIR, "| tồn tại:", OLIST_DIR.exists())

# Tên ngắn -> tên file thật. Đặt tên ngắn để các cell sau gõ cho gọn.
FILES = {
    "customers":   "olist_customers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "items":       "olist_order_items_dataset.csv",
    "payments":    "olist_order_payments_dataset.csv",
    "reviews":     "olist_order_reviews_dataset.csv",
    "orders":      "olist_orders_dataset.csv",
    "products":    "olist_products_dataset.csv",
    "sellers":     "olist_sellers_dataset.csv",
    "translation": "product_category_name_translation.csv",
}

tbl = {}                                        # từ điển: tên ngắn -> DataFrame
for name, fname in FILES.items():
    # dtype=str: đọc mọi cột thành chữ, để mã zip (vd 01310) không mất số 0 đầu.
    tbl[name] = pd.read_csv(OLIST_DIR / fname, dtype=str)
    print(f"{name:12} {len(tbl[name]):>9,} dòng")

thư mục D:\BI\projects\ai-logistics\data\olist | tồn tại: True
customers       99,441 dòng
geolocation  1,000,163 dòng
items          112,650 dòng
payments       103,886 dòng
reviews         99,224 dòng
orders          99,441 dòng
products        32,951 dòng
sellers          3,095 dòng
translation         71 dòng


In [ ]:
# Cell 2
# Mục đích: xem tên cột đầy đủ của 9 bảng (Ngày 1 bị cắt ảnh ở orders và products).
for name, df in tbl.items():
    print(f"{name} ({len(df.columns)} cột)")
    print("   ", df.columns.tolist())           # .tolist() đổi sang list để in một dòng

customers (5 cột)
    ['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']
geolocation (5 cột)
    ['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng', 'geolocation_city', 'geolocation_state']
items (7 cột)
    ['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']
payments (5 cột)
    ['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']
reviews (7 cột)
    ['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp']
orders (8 cột)
    ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']
products (9 cột)
    ['product_id', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qt

In [20]:
# Cell 3
# Mục đích: kiểm khóa dự đoán của mỗi bảng có duy nhất không (kiểm grain).
CANDIDATES = [
    ("customers",   ["customer_id"]),
    ("orders",      ["order_id"]),
    ("items",       ["order_id", "order_item_id"]),
    ("payments",    ["order_id", "payment_sequential"]),
    ("reviews",     ["review_id"]),
    ("reviews",     ["review_id", "order_id"]),
    ("products",    ["product_id"]),
    ("sellers",     ["seller_id"]),
    ("translation", ["product_category_name"]),
    ("geolocation", ["geolocation_zip_code_prefix"]),
]

for table_name, key_cols in CANDIDATES:
    df = tbl[table_name]

    # Bước 1: gom các cột khóa mà bảng không có (gõ sai tên cột sẽ rơi vào đây)
    missing_cols = []                       # danh sách rỗng, sẽ ghi tên cột bị thiếu vào đây
    for col in key_cols:                    # xét từng cột trong khóa
        if col not in df.columns:           # cột này không có trong bảng?
            missing_cols.append(col)        # thì ghi tên nó vào danh sách

    # Bước 2: có cột thiếu thì báo rồi bỏ qua bảng này, không để cell nổ lỗi đỏ
    if len(missing_cols) > 0:
        print(f"{table_name:12} missing columns {missing_cols}, check Cell 2")
        continue                            # nhảy sang bảng kế tiếp

    # Bước 3: đếm số dòng trùng khóa
    duplicates = df.duplicated(subset=key_cols).sum()
    print(f"{table_name:12} {str(key_cols):45} rows {len(df):>9,} | duplicates {duplicates:>9,}")

customers    ['customer_id']                               rows    99,441 | duplicates         0
orders       ['order_id']                                  rows    99,441 | duplicates         0
items        ['order_id', 'order_item_id']                 rows   112,650 | duplicates         0
payments     ['order_id', 'payment_sequential']            rows   103,886 | duplicates         0
reviews      ['review_id']                                 rows    99,224 | duplicates       814
reviews      ['review_id', 'order_id']                     rows    99,224 | duplicates         0
products     ['product_id']                                rows    32,951 | duplicates         0
sellers      ['seller_id']                                 rows     3,095 | duplicates         0
translation  ['product_category_name']                     rows        71 | duplicates         0
geolocation  ['geolocation_zip_code_prefix']               rows 1,000,163 | duplicates   981,148


In [21]:
# Cell 4
# Mục đích: kiểm xem mỗi giá trị khóa ở bảng con có tồn tại ở bảng cha không.
# Giá trị nào không có ở bảng cha = "dòng mồ côi" -> khi nối (JOIN) sẽ bị hụt.

# Mỗi phần tử: (bảng con, cột ở bảng con, bảng cha, cột ở bảng cha)
LINKS = [
    ("items",    "order_id",              "orders",      "order_id"),
    ("items",    "product_id",            "products",    "product_id"),
    ("items",    "seller_id",             "sellers",     "seller_id"),
    ("orders",   "customer_id",           "customers",   "customer_id"),
    ("payments", "order_id",              "orders",      "order_id"),
    ("reviews",  "order_id",              "orders",      "order_id"),
    ("products", "product_category_name", "translation", "product_category_name"),
]

for child, ccol, parent, pcol in LINKS:

    # Bước 1: lấy cột khóa của bảng con (cần kiểm) và của bảng cha (để đối chiếu)
    child_values = tbl[child][ccol]                      # cột khóa ở bảng con
    parent_values = tbl[parent][pcol]                    # cột khóa ở bảng cha

    # Bước 2: đếm NULL ở bảng con (NULL không phải mồ côi, đếm riêng)
    n_null = child_values.isna().sum()

    # Bước 3: bỏ NULL đi, chỉ giữ các giá trị có thật để đối chiếu
    child_not_null = child_values.dropna()

    # Bước 4: mỗi giá trị của bảng con có nằm trong bảng cha không? -> True / False
    found_in_parent = child_not_null.isin(parent_values)

    # Bước 5: đếm số giá trị False = không tìm thấy ở bảng cha = mồ côi
    n_orphan = (found_in_parent == False).sum()

    print(f"{child}.{ccol} -> {parent}.{pcol}: NULL {n_null:,} | mồ côi {n_orphan:,}")

items.order_id -> orders.order_id: NULL 0 | mồ côi 0
items.product_id -> products.product_id: NULL 0 | mồ côi 0
items.seller_id -> sellers.seller_id: NULL 0 | mồ côi 0
orders.customer_id -> customers.customer_id: NULL 0 | mồ côi 0
payments.order_id -> orders.order_id: NULL 0 | mồ côi 0
reviews.order_id -> orders.order_id: NULL 0 | mồ côi 0
products.product_category_name -> translation.product_category_name: NULL 610 | mồ côi 13


In [26]:
# Cell 5
# Mục đích: biết dữ liệu trải từ ngày nào đến ngày nào, và các trạng thái đơn có thật.
orders = tbl["orders"]
ts = pd.to_datetime(orders["order_purchase_timestamp"], errors="coerce")  # lỗi thì thành NaT
print("Đơn không đọc được ngày:", ts.isna().sum())
print("Ngày đầu :", ts.min())
print("Ngày cuối:", ts.max())
print()
print(ts.dt.to_period("M").value_counts().sort_index().to_string())     # số đơn theo tháng
print()
print(orders["order_status"].value_counts(dropna=False))                # trạng thái thật

Đơn không đọc được ngày: 0
Ngày đầu : 2016-09-04 21:15:19
Ngày cuối: 2018-10-17 17:30:18

order_purchase_timestamp
2016-09       4
2016-10     324
2016-12       1
2017-01     800
2017-02    1780
2017-03    2682
2017-04    2404
2017-05    3700
2017-06    3245
2017-07    4026
2017-08    4331
2017-09    4285
2017-10    4631
2017-11    7544
2017-12    5673
2018-01    7269
2018-02    6728
2018-03    7211
2018-04    6939
2018-05    6873
2018-06    6167
2018-07    6292
2018-08    6512
2018-09      16
2018-10       4
Freq: M

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64
